Which user flagged the most distinct videos that ended up approved by YouTube? Output, in one column, their full name or names in case of a tie. In the user's full name, include a space between the first and the last name.
- Tables
  - `user_flags` and `flag_review`
---

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, DateType, BooleanType
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank, concat, lit, countDistinct
import os

In [2]:
problem_name = os.path.abspath(".").split("/")[-1]

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 22:13:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
user_flags_schema = StructType(
    [
        StructField("user_firstname", StringType(), False),
        StructField("user_lastname", StringType(), False),
        StructField("video_id", StringType(), False),
        StructField("flag_id", StringType(), False)
    ]
)

flag_review_schema = StructType(
    [
        StructField("flag_id", StringType(), False),
        StructField("reviewed_by_yt", BooleanType(), False),
        StructField("reviewed_date", DateType(), False),
        StructField("reviewed_outcome", StringType(), False),
    ]
)

In [5]:
user_flags = spark.read\
.option("header", "true")\
.format("csv")\
.schema(user_flags_schema)\
.load("files/user_flags.txt")

In [6]:
user_flags.createOrReplaceTempView("user_flags")

In [7]:
flag_review = spark.read\
.option("header", "true")\
.format("csv")\
.schema(flag_review_schema)\
.load("files/flag_review.txt")

In [8]:
flag_review.createOrReplaceTempView("flag_review")

In [9]:
result_df = flag_review.filter(col("reviewed_outcome")=="APPROVED")\
.join(user_flags, on = (flag_review.flag_id == user_flags.flag_id), how = "inner")\
.withColumn("username", concat(col("user_firstname"), lit(" "), col("user_lastname")))\
.groupBy(col("username")).agg(countDistinct(col("video_id")).alias("unique_video_count"))\
.withColumn("rank_by_video_count", dense_rank().over(Window.orderBy(col("unique_video_count").desc())))\
.filter(col("rank_by_video_count") == 1)\
.select("username")

In [10]:
result_df.toPandas()

26/04/29 22:14:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 2

,username
0,Richard Hasson
1,Mark May


In [11]:
spark.sql(
    """
    with cte as (
    select concat(u.user_firstname, ' ', u.user_lastname) as username, count(distinct video_id) as video_counts
    from user_flags u inner join flag_review f on u.flag_id = f.flag_id
    where f.reviewed_outcome = 'APPROVED'
    group by 1
    ),
    cte2 as (
    select username, dense_rank() over (order by video_counts desc) as rank_by_videos
    from cte 
    )
    select username from cte2 where rank_by_videos = 1
    """
).show()

26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 22:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 2

+--------------+
|      username|
+--------------+
|Richard Hasson|
|      Mark May|
+--------------+

